# Software (SW) Cartoonization Pipeline Benchmark (ARM / OpenCV)

This notebook benchmarks the **software implementation** of the *cartoonization* image-processing pipeline running on the **PYNQ-Z2 on-board ARM processor (PS)** using **Python + OpenCV**.

The goal is to obtain **software latency and performance metrics** that can be compared against the **hardware-accelerated (FPGA/PL)** version of the same pipeline.

---

## 1. What the pipeline does

Given an input color image (BGR), the pipeline produces a “cartoonized” output by combining:
- **Smoothed colors** (via a bilateral filter), and
- **Binary edges** (via adaptive thresholding on a median-blurred grayscale image)

### Processing stages (in order)
1. **Grayscale conversion**  
   Converts the input BGR image to grayscale.
2. **Median blur**  
   Reduces noise while preserving edges (helps edge detection).
3. **Adaptive threshold**  
   Computes a binary “edge mask” from the blurred grayscale image.
4. **Bilateral filter**  
   Smooths colors while preserving boundaries (edge-aware smoothing).
5. **Combine (masking)**  
   Applies the edge mask to the smoothed color image to create the cartoon effect.

---

## 2. What this notebook measures

The notebook runs the pipeline multiple times (with optional warm-up) and records timing for:
- The **entire pipeline** per run
- Each **individual stage** per run

It also captures basic **process resource metrics** for each run (CPU time and memory usage).

---

## 3. Meaning of the metrics

### Timing metrics (seconds)
- **`total_s`**  
  *Wall-clock latency* of one full pipeline run, measured with a high-resolution timer.  
  This is the **end-to-end compute time** of the software pipeline call (excluding file I/O unless you explicitly time image load/save).

- **Stage times** (e.g., `1_grayscale_s`, `2_median_s`, …)  
  Wall-clock times for each pipeline stage.  
  These help identify which stage dominates runtime (often the bilateral filter).

- **`fps`**  
  A throughput-style estimate computed as:
  \[
  \text{fps} = \frac{1}{\text{total\_s}}
  \]
  This reflects how many frames per second the pipeline *could* process if repeated continuously with the same frame size and parameters.

### CPU / memory metrics
- **`cpu_time_delta_s`**  
  CPU time consumed by the Python process during one run (**user + system CPU time**).  
  This is **not** wall-clock time; it reflects “how much CPU work” the process used.

  ⚠️ On multi-core systems and multi-threaded libraries (OpenCV), `cpu_time_delta_s` can be **larger than `total_s`** because CPU time is **summed across threads/cores**.

- **`rss_end_bytes`**  
  Resident Set Size (RSS) at the end of the run — roughly the amount of RAM the process is using.

- **`rss_delta_bytes`**  
  Change in RSS during the run. A large positive value may indicate extra allocations or caching.

---

## 4. Notes for fair SW vs HW comparison

- **SW latency** here reflects computation on the **ARM processor (PS)** using OpenCV, focus on **Compute-only latency** (exlcuding frame transfers).

---

## 5. Inputs / outputs

- Input image is expected to be located in the notebook directory (e.g., `delft.png`).
- Output cartoon images and a CSV log of per-run metrics are saved under:
  `output_opencv/SW_complete_pipeline/`

---


## 1. Imports + paths

In [7]:
import os
import time
import csv
from pathlib import Path

import cv2
import psutil
import numpy as np

# Current notebook directory (in your screenshot: /home/xilinx/jupyter_notebooks/video_cartoonize)
ROOT = Path.cwd()

# Input image you uploaded (highlighted in your screenshot)
INPUT_PATH = ROOT / "delft.png"
assert INPUT_PATH.exists(), f"Input not found: {INPUT_PATH}"

# Output directory
STAGE_NAME = "SW_complete_pipeline"
OUTPUT_DIR = ROOT / "output_opencv" / STAGE_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT      :", ROOT)
print("INPUT_PATH:", INPUT_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


ROOT      : /home/xilinx/jupyter_notebooks/video_cartoonize
INPUT_PATH: /home/xilinx/jupyter_notebooks/video_cartoonize/delft.png
OUTPUT_DIR: /home/xilinx/jupyter_notebooks/video_cartoonize/output_opencv/SW_complete_pipeline


## 2. Helpers (load, metrics snapshot, CSV logger)

In [8]:
proc = psutil.Process(os.getpid())

def load_image_bgr(path: Path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"Failed to read image: {path}")
    return img

def snapshot_process(proc: psutil.Process):
    """
    Returns a dict with process CPU time (user+sys) and RSS.
    """
    ct = proc.cpu_times()
    mem = proc.memory_info()
    return {
        "cpu_time_s": float(ct.user + ct.system),
        "rss_bytes": int(mem.rss),
    }

def write_csv_row(csv_path: Path, row: dict):
    new_file = not csv_path.exists()
    with open(csv_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if new_file:
            w.writeheader()
        w.writerow(row)

CSV_PATH = OUTPUT_DIR / "sw_metrics.csv"


## 3. SW pipeline

In [9]:
# Pipeline parameters (copied from complete_opencv.py)
MEDIAN_K = 5
ADAPT_BLOCK = 5
ADAPT_C = 7
BIL_D = 5
BIL_SIGMA = 100

def sw_cartoon_pipeline(img_bgr):
    t = {}

    t0 = time.perf_counter()
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    t["1_grayscale_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    gray_blur = cv2.medianBlur(gray, MEDIAN_K)
    t["2_median_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    edges = cv2.adaptiveThreshold(
        gray_blur,
        255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY,
        ADAPT_BLOCK,
        ADAPT_C
    )
    t["3_adapt_thresh_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    bil_blurred = cv2.bilateralFilter(img_bgr, BIL_D, BIL_SIGMA, BIL_SIGMA)
    t["4_bilateral_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    cartoon = cv2.bitwise_and(bil_blurred, bil_blurred, mask=edges)
    t["5_combine_s"] = time.perf_counter() - t0

    return cartoon, t


## 4. Run N times (warmup + measured runs), save output, log metrics

In [10]:
img = load_image_bgr(INPUT_PATH)

# --- Warmup (For more stable measurements) ---
for _ in range(2):
    _ = sw_cartoon_pipeline(img)

RUNS = 10

results = []
for i in range(RUNS):
    snap0 = snapshot_process(proc)

    t_start = time.perf_counter()
    out, stage_t = sw_cartoon_pipeline(img)
    t_end = time.perf_counter()

    snap1 = snapshot_process(proc)

    total_s = t_end - t_start
    fps = 1.0 / total_s if total_s > 0 else float("inf")

    out_name = f"sw_cartoon_run{i:02d}.png"
    out_path = OUTPUT_DIR / out_name
    cv2.imwrite(str(out_path), out)

    row = {
        "run": i,
        "input": str(INPUT_PATH.name),
        "output": out_name,
        "total_s": total_s,
        "fps": fps,
        **stage_t,
        "cpu_time_delta_s": snap1["cpu_time_s"] - snap0["cpu_time_s"],
        "rss_delta_bytes": snap1["rss_bytes"] - snap0["rss_bytes"],
        "rss_end_bytes": snap1["rss_bytes"],
    }

    results.append(row)
    write_csv_row(CSV_PATH, row)

print(f"Done. Wrote per-run metrics to: {CSV_PATH}")
print(f"Last output saved to: {out_path}")


Done. Wrote per-run metrics to: /home/xilinx/jupyter_notebooks/video_cartoonize/output_opencv/SW_complete_pipeline/sw_metrics.csv
Last output saved to: /home/xilinx/jupyter_notebooks/video_cartoonize/output_opencv/SW_complete_pipeline/sw_cartoon_run09.png


## 5. Summary (mean / std latency, stage breakdown)

In [11]:
def mean_std(vals):
    vals = np.array(vals, dtype=float)
    return float(vals.mean()), float(vals.std(ddof=1)) if len(vals) > 1 else 0.0

total_mean, total_std = mean_std([r["total_s"] for r in results])
fps_mean, fps_std = mean_std([r["fps"] for r in results])

print("=== SW Pipeline Latency Summary (ARM + OpenCV) ===")
print(f"Runs: {RUNS}")
print(f"Total latency: mean={total_mean*1e3:.3f} ms, std={total_std*1e3:.3f} ms")
print(f"FPS estimate:  mean={fps_mean:.2f}, std={fps_std:.2f}")

stage_keys = [k for k in results[0].keys() if k.endswith("_s") and k != "total_s"]
print("\n--- Per-stage mean latency (ms) ---")
for k in stage_keys:
    m, s = mean_std([r[k] for r in results])
    print(f"{k:18s}: {m*1e3:8.3f} ms  (std {s*1e3:6.3f} ms)")


=== SW Pipeline Latency Summary (ARM + OpenCV) ===
Runs: 10
Total latency: mean=2333.031 ms, std=84.376 ms
FPS estimate:  mean=0.43, std=0.01

--- Per-stage mean latency (ms) ---
1_grayscale_s     :   10.933 ms  (std  0.056 ms)
2_median_s        :  406.009 ms  (std 12.119 ms)
3_adapt_thresh_s  :   49.036 ms  (std  8.114 ms)
4_bilateral_s     : 1825.092 ms  (std 68.485 ms)
5_combine_s       :   38.369 ms  (std  0.071 ms)
cpu_time_delta_s  : 4075.000 ms  (std 31.358 ms)
